# ComplianceGPT — Pipeline Acceptance Tests

This notebook is a **regression gate** for the ComplianceGPT pipeline.

It checks:
- **Final Answer Contract schema** (required fields exist)
- **Status ↔ evidence consistency invariants**
- Basic **PARAMS_REQUIRED** and **NO_EVIDENCE** behavioral constraints (non-brittle)

It does **not** measure retrieval quality. Use batch evaluation notebooks for that.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


Mounted at /content/drive


In [1]:
import os, warnings

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")


In [3]:
import os, sys

PROJECT_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"
assert os.path.isdir(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import compliancegpt
print("compliancegpt.__file__ =", compliancegpt.__file__)


compliancegpt.__file__ = None


In [4]:
# --- Canonical paths (Drive) ---
import os

DATA_DIR = os.path.join(PROJECT_ROOT, "data")

REV5_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev5_catalog.jsonl")
REV4_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev4_catalog.jsonl")

REV5_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev5_gold-set_100q.csv")
REV4_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev4_gold-set_36q.csv")

ORG_PROFILE_REV5 = os.path.join(DATA_DIR, "ODP", "rev5", "org_profile_example_rev5.yaml")
ORG_PROFILE_REV4 = os.path.join(DATA_DIR, "ODP", "rev4", "org_profile_example_rev4.yaml")


In [5]:
# --- load gold sets ---
from typing import Any, Dict, Optional
import pandas as pd

def load_gold(framework_version: str) -> pd.DataFrame:
    path = REV5_GOLD_PATH if framework_version.lower() == "rev5" else REV4_GOLD_PATH
    # use latin1 for robustness.
    df = pd.read_csv(path, encoding="latin1")

    # Standardize columns to pipeline-friendly names
    if "id" in df.columns and "query_id" not in df.columns:
        df["query_id"] = df["id"].astype(str)
    if "question" in df.columns and "original_query" not in df.columns:
        df["original_query"] = df["question"].astype(str)
    return df

def get_gold_row(df: pd.DataFrame, qid: Any) -> Dict[str, Any]:
    qid_str = str(qid)
    m = df[df["query_id"].astype(str) == qid_str]
    if m.empty:
        raise KeyError(f"query_id not found in gold set: {qid_str}")
    return dict(m.iloc[0].to_dict())


In [6]:
# --- Build pipeline (single place to configure knobs) ---
from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline

def build_pipeline(
    *,
    framework_version: str,
    use_org_profile: bool,
    verify_strict_extras: bool = False,
    verify_strict_verbatim: bool = True,
    verify_strict_version: bool = False,
) -> ComplianceGPTPipeline:
    fw = framework_version.lower().strip()
    if fw not in {"rev4", "rev5"}:
        raise ValueError(f"framework_version must be rev4 or rev5 (got {framework_version!r})")

    ccs_path = REV5_CATALOG_PATH if fw == "rev5" else REV4_CATALOG_PATH

    org_profile = None
    if use_org_profile:
        org_profile = ORG_PROFILE_REV5 if fw == "rev5" else ORG_PROFILE_REV4
        if not os.path.exists(org_profile):
            print(f"[WARN] org_profile not found; continuing without profile: {org_profile}")
            org_profile = None

    pipe = ComplianceGPTPipeline(
        framework_version=fw,
        ccs_path=ccs_path,
        use_qur=False,                 # acceptance tests: keep deterministic
        org_profile_path=org_profile,  # optional
        verify_strict_extras=bool(verify_strict_extras),
        verify_strict_verbatim=bool(verify_strict_verbatim),
        verify_strict_version=bool(verify_strict_version),
    )
    return pipe


[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib


In [7]:
# === USER CONFIG ===
FRAMEWORK_VERSION = "rev5"   # "rev4" or "rev5"
USE_GENERATOR = True         # True exercises contract construction; False isolates retrieval

USE_ORG_PROFILE = False      # Off by default for acceptance tests

VERIFY_STRICT_EXTRAS = False
VERIFY_STRICT_VERBATIM = True
VERIFY_STRICT_VERSION = False

TOP_K = 12


In [8]:
# --- Config pipeline ---
pipe = build_pipeline(
    framework_version=FRAMEWORK_VERSION,
    use_org_profile=USE_ORG_PROFILE,
    verify_strict_extras=VERIFY_STRICT_EXTRAS,
    verify_strict_verbatim=VERIFY_STRICT_VERBATIM,
    verify_strict_version=VERIFY_STRICT_VERSION,
)
print("[OK] Pipeline initialized.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[OK] Pipeline initialized.


In [9]:
# --- Acceptance assertions (contract invariants) ---
from typing import List

def assert_contract_schema(c: Dict[str, Any]) -> None:
    required = [
        "answer_text",
        "evidence_spans",
        "status",
        "odp_required_list",
        "primary_citation",
        "all_citations",
        "answer_text_with_citation",
        "contract_mode",
    ]
    missing = [k for k in required if k not in c]
    assert not missing, f"Missing required contract fields: {missing}"
    assert c.get("contract_mode") == "provably_extractive", f"contract_mode mismatch: {c.get('contract_mode')}"
    assert isinstance(c.get("evidence_spans"), list), "evidence_spans must be a list"

def assert_status_consistency(c: Dict[str, Any]) -> None:
    status = str(c.get("status", "")).upper().strip()
    spans = c.get("evidence_spans") or []
    if status == "NO_EVIDENCE":
        assert len(spans) == 0, "NO_EVIDENCE must have empty evidence_spans"
    if status in {"OK", "PARAMS_REQUIRED"}:
        assert len(spans) >= 1, f"{status} must have at least 1 evidence span"

def run_one(query: str, gold_row: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    # IMPORTANT: verifier should only be requested when gold labels exist.
    run_verify = bool(gold_row is not None)

    out = pipe.answer(
        query,
        top_k=int(TOP_K),
        rewrites=None,
        gold_row=gold_row,
        use_generator=bool(USE_GENERATOR),
        run_verify=run_verify,
    )
    c = out.get("contract", out)

    assert_contract_schema(c)
    assert_status_consistency(c)
    return c


## Test set

In [10]:
# --- Select test cases from gold (stable IDs) ---
gold_df = load_gold(FRAMEWORK_VERSION)

# Test-1 (ODP-related): ID 1 is known to be ODP-related in the provided sets.
gold_row_params = get_gold_row(gold_df, 1)
query_params = str(gold_row_params["original_query"]).strip()
print("[T1] PARAMS_REQUIRED candidate query:", query_params)

# Test-3 (OK-ish): choose a simple early ID; no strong retrieval-quality expectation.
try:
    gold_row_ok = get_gold_row(gold_df, 3)
    query_ok = str(gold_row_ok["original_query"]).strip()
except Exception:
    gold_row_ok = None
    query_ok = "What is the principle of least privilege?"


[T1] PARAMS_REQUIRED candidate query: When must account managers and the designated personnel be notified about user account changes?


In [11]:
# --- 1) PARAMS_REQUIRED behavior (non-brittle) ---
c1 = run_one(query_params, gold_row=gold_row_params)

s1 = str(c1.get("status","")).upper().strip()
sp1 = c1.get("evidence_spans") or []
odps1 = c1.get("odp_required_list") or []

print(f"[T1] status={s1} evidence_spans_len={len(sp1)} odp_required_list_len={len(odps1)}")

# Hard requirement: if pipeline claims PARAMS_REQUIRED, it must list required params.
if s1 == "PARAMS_REQUIRED":
    assert len(odps1) >= 1, "PARAMS_REQUIRED must have non-empty odp_required_list"


[T1] status=PARAMS_REQUIRED evidence_spans_len=3 odp_required_list_len=4


In [12]:
# --- 2) NO_EVIDENCE behavior (nonsense query; consistency only) ---
nonsense_query = "ZZZ-NONEXISTENT-CONTROL-ID: this query should not match anything"
c2 = run_one(nonsense_query, gold_row=None)

s2 = str(c2.get("status","")).upper().strip()
sp2 = c2.get("evidence_spans") or []

print(f"[T2] status={s2} evidence_spans_len={len(sp2)} (NO_EVIDENCE is allowed but not guaranteed)")


[T2] status=NO_EVIDENCE evidence_spans_len=0 (NO_EVIDENCE is allowed but not guaranteed)


In [13]:
# --- 3) Schema stability on an OK-ish gold question ---
c3 = run_one(query_ok, gold_row=gold_row_ok)

s3 = str(c3.get("status","")).upper().strip()
sp3 = c3.get("evidence_spans") or []
odps3 = c3.get("odp_required_list") or []

print(f"[T3] status={s3} evidence_spans_len={len(sp3)} odp_required_list_len={len(odps3)}")

# --- Summary ---
summary = {
    "T1_status": s1,
    "T2_status": s2,
    "T3_status": s3,
    "use_generator": bool(USE_GENERATOR),
    "framework_version": str(FRAMEWORK_VERSION),
}
print("\n=== Acceptance Test Summary ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n[OK] Acceptance tests completed.")


[T3] status=OK evidence_spans_len=2 odp_required_list_len=0

=== Acceptance Test Summary ===
T1_status: PARAMS_REQUIRED
T2_status: NO_EVIDENCE
T3_status: OK
use_generator: True
framework_version: rev5

[OK] Acceptance tests completed.
